In [1]:
import pandas as pd
import mysql.connector
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
# List of CSV files and their corresponding table names
csv_files = [
    ('customers.csv', 'customers'),
    ('orders.csv', 'orders'),
    ('sellers.csv', 'sellers'),
    ('products.csv', 'products'),
    ('geolocation.csv', 'geolocation'),
    ('payments.csv', 'payments'),
    ('order_items.csv','order_items')
    # Added payments.csv for specific handling
]

# Connect to the MySQL database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='',
    database='ecommerce'
)
cursor = conn.cursor()

# Folder containing the CSV files
folder_path = 'C:/Users/Dell/Downloads/Data Science/Project/Ecommerce Analysis'

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'TEXT'

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
    
    # Replace NaN with None to handle SQL NULL
    df = df.where(pd.notnull(df), None)
    
    # Debugging: Check for NaN values
    print(f"Processing {csv_file}")
    print(f"NaN values before replacement:\n{df.isnull().sum()}\n")

    # Clean column names
    df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') for col in df.columns]

    # Generate the CREATE TABLE statement with appropriate data types
    columns = ', '.join([f'`{col}` {get_sql_type(df[col].dtype)}' for col in df.columns])
    create_table_query = f'CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})'
    cursor.execute(create_table_query)

    # Insert DataFrame data into the MySQL table
    for _, row in df.iterrows():
        # Convert row to tuple and handle NaN/None explicitly
        values = tuple(None if pd.isna(x) else x for x in row)
        sql = f"INSERT INTO `{table_name}` ({', '.join(['`' + col + '`' for col in df.columns])}) VALUES ({', '.join(['%s'] * len(row))})"
        cursor.execute(sql, values)

    # Commit the transaction for the current CSV file
    conn.commit()

# # Close the connection
# conn.close()

Processing customers.csv
NaN values before replacement:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Processing orders.csv
NaN values before replacement:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Processing sellers.csv
NaN values before replacement:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Processing products.csv
NaN values before replacement:
product_id                      0
product category              610
product_name_length           610
product_description_length    610
product_photos_qty            610
prod

In [ ]:
# cursor = conn.cursor(buffered=True)


In [5]:
# Basic Queries
# 1. List all unique cities where customers are located.
# 2. Count the number of orders placed in 2017.
# 3. Find the total sales per category.
# 4. Calculate the percentage of orders that were paid in installments.
# 5. Count the number of customers from each state. 


In [6]:
# 1. List all unique cities where customers are located.

In [ ]:
query = """ select distinct(customer_city) from customers"""
cursor.execute(query)
data = cursor.fetchall()
data

In [4]:
#2. Count the number of orders placed in 2017.

In [ ]:
query = """ select count(order_id) from orders where year(order_purchase_timestamp ) = 2017"""
cursor.execute(query)
data = cursor.fetchall()
"total orders placed in 2017 are", data[0][0]


In [6]:
# 3. Find the total sales per category.


In [ ]:
query = """ select product_category,round(sum(t1.payment_value),2) 
from payments t1 join order_items t2 on t1.order_id = t2.order_id join 
products t3 on t2.product_id = t3.product_id
group by product_category """

cursor.execute(query)
data = cursor.fetchall()
df = pd.DataFrame(data,columns = ["Category","Sales"])
df

In [ ]:
#4. Calculate the percentage of orders that were paid in installments.

In [ ]:
query = """ select (sum(case when payment_installments > 0 then 1 else 0 end))/count(*)* 100 from payments """

cursor.execute(query)
data = cursor.fetchall()
data

In [ ]:
#5. Count the number of customers from each state. 

In [ ]:
query = """ select customer_state , count(*) from customers group by customer_state """

cursor.execute(query)
data = cursor.fetchall()
data 
df = pd.DataFrame(data,columns = ["Customer State","Total Customers"])
df = df.sort_values(by = "Total Customers", ascending = False)
plt.figure(figsize = (12,6))
plt.bar(df["Customer State"],df["Total Customers"])

In [ ]:
# Intermediate Queries
# 1. Calculate the number of orders per month in 2018.
# 2. Find the average number of products per order, grouped by customer city.
# 3. Calculate the percentage of total revenue contributed by each product category.
# 4. Identify the correlation between product price and the number of times a product has been purchased.
# 5. Calculate the total revenue generated by each seller, and rank them by revenue.


In [ ]:
# 1. Calculate the number of orders per month in 2018.

In [3]:

query = """ SELECT 
    MONTHNAME(order_purchase_timestamp) AS month,
    COUNT(order_id) AS total_orders
FROM orders
WHERE YEAR(order_purchase_timestamp) = 2018
GROUP BY 
    MONTH(order_purchase_timestamp),
    MONTHNAME(order_purchase_timestamp)
ORDER BY MONTH(order_purchase_timestamp) """

cursor.execute(query)
data = cursor.fetchall()
data 
# df = pd.DataFrame(df,columns = ["",""])


[('January', 43614),
 ('February', 40368),
 ('March', 43266),
 ('April', 41634),
 ('May', 41238),
 ('June', 37002),
 ('July', 37752),
 ('August', 39072),
 ('September', 96),
 ('October', 24)]

In [4]:
#2. Find the average number of products per order, grouped by customer city.

In [7]:
query = """ with count_per_order as
(select orders.order_id,orders.customer_id,
count(order_items.product_id) as 'oc' from orders join order_items
 on orders.order_id = order_items.order_id
 group by orders.order_id,orders.customer_id)

select customers.customer_city , 
round(avg(count_per_order.oc),2)from customers join count_per_order
on customers.customer_id = count_per_order.customer_id
group by customers.customer_city """

cursor.execute(query)
data = cursor.fetchall()
data

df = pd.DataFrame(data,columns = ["City","Average Orders"])
df

,City,Average Orders
0,sao paulo,41.62
1,sao jose dos campos,40.99
2,porto alegre,42.30
3,indaial,40.15
4,treze tilias,45.82
...,...,...
4105,sambaiba,36.00
4106,guairaca,72.00
4107,japaratuba,36.00
4108,tuiuti,36.00


In [9]:
#3. Calculate the percentage of total revenue contributed by each product category.

In [12]:
query = """ SELECT 
    product_category,
    round((SUM(t1.payment_value) / (SELECT 
            SUM(payment_value)
        FROM
            payments)) * 100,2) as 'sales'
FROM
    payments t1
        JOIN
    order_items t2 ON t1.order_id = t2.order_id
        JOIN
    products t3 ON t2.product_id = t3.product_id
GROUP BY product_category
order by sales desc """

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data,columns = ["CATEGORY","PERCENTAGE CONTRIBUTION TO TOTAL REVENUE"])
df

,CATEGORY,PERCENTAGE CONTRIBUTION TO TOTAL REVENUE
0,bed table bath,385.11
1,HEALTH BEAUTY,372.70
2,computer accessories,356.50
3,Furniture Decoration,321.61
4,Watches present,321.40
...,...,...
69,PC Gamer,0.49
70,House Comfort 2,0.38
71,cds music dvds,0.27
72,Fashion Children's Clothing,0.18


In [3]:
# 4.Identify the correlation between product price and the number of times a product has been purchased.

In [14]:
query = """ select t1.product_category,count(t2.product_id),avg(t2.price)
from products t1 join order_items t2 on 
t1.product_id = t2.product_id 
group by t1.product_category """

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data,columns = ["Category","Order_Counts","Price"])
df.head()
df.corr(numeric_only=True)


,Order_Counts,Price
Order_Counts,1.000000,-0.106316
Price,-0.106316,1.000000


In [ ]:
# 5. Calculate the total revenue generated by each seller, and rank them by revenue.

In [24]:
query = """ SELECT *,
       DENSE_RANK() OVER(ORDER BY revenue DESC) AS seller_rank
FROM
(
    SELECT 
        t1.seller_id,
        ROUND(SUM(t2.payment_value),2) AS revenue
    FROM order_items t1
    JOIN payments t2
        ON t1.order_id = t2.order_id
    GROUP BY t1.seller_id
) AS seller_revenue;
"""

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data,columns = ["Seller Id","Revenue","Rank"])
df

,Seller Id,Revenue,Rank
0,7c67e1448b00f6e969d365cea6b010ab,32458682.07,1
1,1025f0e2d44d7041d6cf58b6550e0bfa,19726210.55,2
2,4a3ca9315b744ce9f8e9374361493884,19279697.26,3
3,1f50f920176fa81dab994f9023523100,18576218.89,4
4,53243585a1d6dc2643021fd1853d8905,18233797.15,5
...,...,...,...
3090,ad14615bdd492b01b0d97922e87cb87f,1229.44,3076
3091,702835e4b785b67a084280efca355756,1187.84,3077
3092,4965a7002cca77301c82d3f91b82e1a9,1047.04,3078
3093,77128dec4bec4878c37ab7d6169d6f26,974.08,3079


In [ ]:
# Advanced Queries
# 1. Calculate the moving average of order values for each customer over their order history.
# 2. Calculate the cumulative sales per month for each year.
# 3. Calculate the year-over-year growth rate of total sales.
# 4. Calculate the retention rate of customers, defined as the percentage of customers who make another purchase within 6 months of their first purchase.
# 5. Identify the top 3 customers who spent the most money in each year.


In [25]:
# 1. Calculate the moving average of order values for each customer over their order history.

In [3]:
query = """ select customer_id,order_purchase_timestamp,payment,
avg(payment) over (partition by customer_id order by 
order_purchase_timestamp rows between 2 preceding and 
current row) as mov_avg from
(select orders.customer_id,orders.order_purchase_timestamp,
payments.payment_value as payment
from payments join orders on 
payments.order_id = orders.order_id) as a
;
"""

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data,columns = ["Customer Id","Date","payment","Moving Avg"])
df

# If you remove it:
# AVG(payment) OVER(PARTITION BY customer_id ORDER BY date)
# Then it becomes cumulative average (running average), not moving average.

,Customer Id,Date,payment,Moving Avg
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
1,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
2,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
3,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
4,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
...,...,...,...,...
11219683,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001
11219684,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001
11219685,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001
11219686,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001


In [ ]:
#2. Calculate the cumulative sales per month for each year.

In [6]:
query = """ SELECT years, months, payment, SUM(payment) OVER(ORDER BY years, months) AS cumulative_revenue
FROM
(
    SELECT 
        YEAR(t1.order_purchase_timestamp) AS years,
        MONTH(t1.order_purchase_timestamp) AS months,
        # MONTHNAME(t1.order_purchase_timestamp) AS months,
        ROUND(SUM(t2.payment_value),2) AS payment
    FROM orders t1
    JOIN payments t2 
        ON t1.order_id = t2.order_id
    GROUP BY 
        YEAR(t1.order_purchase_timestamp),
        MONTH(t1.order_purchase_timestamp),
        MONTHNAME(t1.order_purchase_timestamp)
) AS monthly_sales
ORDER BY years, months;
"""

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data,columns = ["Year","Month","Payment","Cumulative Revenue"])
df.head(10)





,Year,Month,Payment,Cumulative Revenue
0,2016,9,27241.92,2.724192e+04
1,2016,10,6381771.84,6.409014e+06
2,2016,12,2118.96,6.411133e+06
3,2017,1,14956708.33,2.136784e+07
4,2017,2,31526065.03,5.289391e+07
5,2017,3,48585268.75,1.014792e+08
6,2017,4,45121107.19,1.466003e+08
7,2017,5,64035232.57,2.106355e+08
8,2017,6,55217849.08,2.658534e+08
9,2017,7,63977355.30,3.298307e+08


In [7]:
#3. Calculate the year-over-year growth rate of total sales.

In [10]:
query = """with a as(select year(t1.order_purchase_timestamp) as years ,sum(t2.payment_value) as payment from orders t1 
join payments t2 on t1.order_id = t2.order_id
group by year(t1.order_purchase_timestamp) order by year(t1.order_purchase_timestamp))

select years,((payment - lag(payment,1) over(order by years)) / lag(payment,1) over(order by years)) * 100 from a """

cursor.execute(query)
data = cursor.fetchall()
data
df = pd.DataFrame(data, columns = ["years","yoy % growth"])

InternalError: Unread result found

In [ ]:
#4. Calculate the retention rate of customers, defined as the percentage of customers who make another purchase within 6 months of their first purchase.

In [ ]:
query = """with a as (select customers.customer_id,
min(orders.order_purchase_timestamp) first_order
from customers join orders
on customers.customer_id = orders.customer_id
group by customers.customer_id),

b as (select a.customer_id, count(distinct orders.order_purchase_timestamp) next_order
from a join orders
on orders.customer_id = a.customer_id
and orders.order_purchase_timestamp > first_order
and orders.order_purchase_timestamp < 
date_add(first_order, interval 6 month)
group by a.customer_id) 

select 100 * (count( distinct a.customer_id)/ count(distinct b.customer_id)) 
from a left join b 
on a.customer_id = b.customer_id ;"""

cur.execute(query)
data = cur.fetchall()

data

In [ ]:
#5. Identify the top 3 customers who spent the most money in each year

In [ ]:
query = """select years, customer_id, payment, d_rank
from
(select year(orders.order_purchase_timestamp) years,
orders.customer_id,
sum(payments.payment_value) payment,
dense_rank() over(partition by year(orders.order_purchase_timestamp)
order by sum(payments.payment_value) desc) d_rank
from orders join payments 
on payments.order_id = orders.order_id
group by year(orders.order_purchase_timestamp),
orders.customer_id) as a
where d_rank <= 3 ;"""

cur.execute(query)
data = cur.fetchall()
df = pd.DataFrame(data, columns = ["years","id","payment","rank"])
sns.barplot(x = "id", y = "payment", data = df, hue = "years")
plt.xticks(rotation = 90)
plt.show()